# Caso de estudio: Segmentación del consumo de café en el trabajo con Red Neuronal

## Objetivo de aprendizaje
En este ejercicio aplicaremos un **modelo descriptivo** sobre el consumo de café en el trabajo, utilizando una **red neuronal no supervisada** para reducir la información y luego agrupar perfiles similares.

Al finalizar, el estudiante debería ser capaz de:

- comprender la diferencia entre un problema **descriptivo** y uno **predictivo**
- cargar un dataset en Google Colab
- analizar variables de consumo de café
- preparar y escalar datos
- entrenar una **red neuronal autoencoder**
- obtener una representación reducida de los datos
- aplicar clustering sobre esa nueva representación
- evaluar si el modelo está operando correctamente con métricas adecuadas
- interpretar los grupos encontrados

---

## Contexto del caso
Una empresa desea comprender los **patrones de consumo de café entre sus trabajadores** para mejorar sus espacios de descanso, definir estrategias de bienestar y entender hábitos laborales.

Para ello dispone de variables como:

- edad
- horas de trabajo al día
- tazas de café al día
- horas de sueño
- nivel de estrés
- cantidad de reuniones diarias
- trabajo remoto

A diferencia de los modelos predictivos, aquí **no existe una variable objetivo a predecir**.

El propósito es:

> **descubrir perfiles o grupos de trabajadores con características similares respecto al consumo de café y su contexto laboral**

Por ello estamos frente a un **modelo descriptivo**.

---

## ¿Por qué usar una red neuronal en un problema descriptivo?
En este caso utilizaremos un **Autoencoder**, que es una red neuronal no supervisada.

Su función no es predecir una respuesta, sino:

- aprender una representación comprimida de los datos
- detectar estructuras internas
- resumir la información relevante
- facilitar posteriormente la agrupación de perfiles

Luego, sobre esa representación comprimida, aplicaremos un algoritmo de agrupamiento.

Esto permite combinar:
- **red neuronal** para aprender patrones latentes
- **clustering** para formar grupos


## Paso 1: Cargar el dataset

En Google Colab primero debemos subir el archivo CSV.

Ejecuta la siguiente celda y selecciona el archivo:

**dataset_consumo_cafe_trabajo.csv**


In [ ]:
from google.colab import files
uploaded = files.upload()

## Paso 2: Importar librerías

Estas librerías nos permitirán:

- manipular datos con **pandas**
- hacer cálculos con **numpy**
- escalar variables con **StandardScaler**
- construir una red neuronal con **TensorFlow / Keras**
- aplicar clustering con **KMeans**
- evaluar el agrupamiento con métricas descriptivas
- visualizar resultados con **matplotlib**


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense

## Paso 3: Leer el archivo CSV

Ahora cargamos el dataset y mostramos sus primeras filas para verificar que los datos se hayan leído correctamente.


In [ ]:
df = pd.read_csv("dataset_consumo_cafe_trabajo.csv")

print("Dimensión del dataset:", df.shape)
df.head()

## Paso 4: Comprender las variables del problema

En este caso trabajaremos con las siguientes variables:

- **EDAD**: edad del trabajador
- **HORAS_TRABAJO_DIA**: horas de trabajo al día
- **TAZAS_CAFE_DIA**: cantidad de tazas de café consumidas al día
- **HORAS_SUENO**: horas promedio de sueño
- **NIVEL_ESTRES**: nivel de estrés reportado
- **REUNIONES_DIA**: cantidad de reuniones por día
- **TRABAJO_REMOTO**: si trabaja remoto o no

### Importante
Aquí no existe una variable objetivo como en los problemas predictivos.  
Todas las variables se analizan al mismo nivel, porque el propósito es encontrar **patrones ocultos**.


In [ ]:
print("Columnas del dataset:")
for c in df.columns:
    print("-", c)

## Paso 5: Revisión general de los datos

Antes de entrenar cualquier modelo debemos revisar:

- tipos de datos
- valores nulos
- estadísticas generales

Esto permite verificar que el dataset esté correctamente preparado.


In [ ]:
print("Información general:")
print(df.info())

print("\nValores nulos por columna:")
print(df.isnull().sum())

print("\nResumen estadístico:")
df.describe()

## Paso 6: Exploración inicial de los datos

En un modelo descriptivo, la exploración inicial ayuda a comprender el comportamiento general de las variables.

Observaremos algunas distribuciones relevantes.


In [ ]:
plt.figure(figsize=(8,5))
plt.hist(df["TAZAS_CAFE_DIA"], bins=6)
plt.title("Distribución de tazas de café por día")
plt.xlabel("Tazas de café")
plt.ylabel("Frecuencia")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(df["NIVEL_ESTRES"], df["TAZAS_CAFE_DIA"])
plt.title("Nivel de estrés vs tazas de café")
plt.xlabel("Nivel de estrés")
plt.ylabel("Tazas de café al día")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(df["HORAS_SUENO"], df["TAZAS_CAFE_DIA"])
plt.title("Horas de sueño vs tazas de café")
plt.xlabel("Horas de sueño")
plt.ylabel("Tazas de café al día")
plt.show()

## Paso 7: Preparar los datos para la red neuronal

Las redes neuronales trabajan mejor cuando las variables están en una escala comparable.

Por eso aplicaremos **estandarización**:

- media cercana a 0
- desviación estándar cercana a 1

Esto evita que una variable con valores grandes domine a las demás.


In [ ]:
X = df.copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Forma original:", X.shape)
print("Forma escalada:", X_scaled.shape)

## Paso 8: ¿Qué es un Autoencoder?

Un **Autoencoder** es una red neuronal que intenta reconstruir su propia entrada.

Tiene dos partes:

1. **Encoder**  
   Reduce la información a una representación más compacta.

2. **Decoder**  
   Reconstruye los datos originales a partir de esa versión comprimida.

### ¿Por qué sirve en este caso?
Porque permite:

- resumir patrones complejos
- reducir ruido
- descubrir estructuras internas
- generar una representación latente útil para segmentar

En este ejercicio el autoencoder aprenderá una versión reducida del comportamiento de los trabajadores.


## Paso 9: Definir la arquitectura de la red neuronal

Usaremos una red simple:

- capa de entrada
- capa oculta intermedia
- capa latente comprimida
- capa de salida

La capa latente será la representación reducida que luego utilizaremos para agrupar trabajadores.


In [ ]:
input_dim = X_scaled.shape[1]
encoding_dim = 3

input_layer = Input(shape=(input_dim,))
encoded = Dense(6, activation='relu')(input_layer)
latent = Dense(encoding_dim, activation='relu', name="capa_latente")(encoded)
decoded = Dense(6, activation='relu')(latent)
output_layer = Dense(input_dim, activation='linear')(decoded)

autoencoder = Model(inputs=input_layer, outputs=output_layer)
encoder = Model(inputs=input_layer, outputs=latent)

autoencoder.compile(optimizer='adam', loss='mse')

autoencoder.summary()

## Paso 10: Entrenar la red neuronal

La red aprenderá a reconstruir los datos originales.

### ¿Cómo sabemos si está aprendiendo?
Observaremos la función de pérdida (**loss**), que en este caso es el **error cuadrático medio de reconstrucción**.

Mientras más baje el loss, mejor está aprendiendo a representar los datos.


In [ ]:
history = autoencoder.fit(
    X_scaled, X_scaled,
    epochs=100,
    batch_size=16,
    validation_split=0.2,
    verbose=0
)

print("Entrenamiento finalizado.")

## Paso 11: Revisar si la red neuronal está operando correctamente

En un autoencoder no usamos accuracy, recall o F1-score, porque no estamos clasificando ni prediciendo etiquetas.

En cambio, revisamos:

### 1. Loss de entrenamiento
Mide qué tan bien la red reconstruye los datos de entrenamiento.

### 2. Validation Loss
Mide qué tan bien reconstruye datos que no usó directamente para ajustar pesos.

### Interpretación
- si ambos valores bajan y se mantienen razonables, la red está aprendiendo
- si validation loss es mucho mayor que training loss, puede haber sobreajuste


In [ ]:
train_loss = history.history["loss"]
val_loss = history.history["val_loss"]

print("Último loss de entrenamiento:", round(train_loss[-1], 4))
print("Último loss de validación:", round(val_loss[-1], 4))

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(train_loss, label="Entrenamiento")
plt.plot(val_loss, label="Validación")
plt.title("Evolución del loss del autoencoder")
plt.xlabel("Épocas")
plt.ylabel("Loss (MSE)")
plt.legend()
plt.show()

## Paso 12: Obtener la representación latente

Ahora usaremos solo la parte **encoder** para transformar los datos originales en una representación más compacta.

Esta será la base para encontrar grupos similares.


In [ ]:
X_latent = encoder.predict(X_scaled, verbose=0)

print("Forma de la representación latente:", X_latent.shape)
pd.DataFrame(X_latent).head()

## Paso 13: Elegir el número de clusters

Ahora aplicaremos un algoritmo de agrupamiento sobre la representación latente.

Usaremos el **método del codo** para observar cómo cambia la inercia según la cantidad de clusters.


In [ ]:
inertia = []

for k in range(1, 10):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_latent)
    inertia.append(km.inertia_)

plt.figure(figsize=(8,5))
plt.plot(range(1,10), inertia, marker='o')
plt.title("Método del codo sobre representación latente")
plt.xlabel("Número de clusters")
plt.ylabel("Inercia")
plt.show()

## Paso 14: Aplicar clustering sobre la representación aprendida

En este ejemplo usaremos **3 clusters**.

### ¿Qué estamos haciendo?
- la red neuronal resume el comportamiento
- KMeans agrupa trabajadores similares sobre esa nueva representación


In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_latent)

df["CLUSTER"] = clusters

df.head()

## Paso 15: Métricas para determinar si el modelo descriptivo está operando correctamente

En clustering y modelos descriptivos, las métricas apropiadas son distintas a las de clasificación.

Usaremos tres métricas importantes:

### 1. Silhouette Score
Mide qué tan bien separados están los grupos.

- cercano a **1** → grupos bien definidos
- cercano a **0** → grupos mezclados
- negativo → mala agrupación

### 2. Davies-Bouldin Index
Mide la similitud entre clusters.

- mientras **menor**, mejor

### 3. Calinski-Harabasz Score
Evalúa separación entre clusters y cohesión interna.

- mientras **mayor**, mejor

Estas métricas permiten justificar si el modelo está agrupando de forma razonable.


In [ ]:
sil_score = silhouette_score(X_latent, clusters)
db_score = davies_bouldin_score(X_latent, clusters)
ch_score = calinski_harabasz_score(X_latent, clusters)

print("MÉTRICAS DEL MODELO DESCRIPTIVO")
print("--------------------------------")
print("Silhouette Score       :", round(sil_score, 4))
print("Davies-Bouldin Index   :", round(db_score, 4))
print("Calinski-Harabasz Score:", round(ch_score, 4))

## Paso 16: Interpretación guiada de las métricas

El siguiente bloque entrega una interpretación básica para apoyar el análisis en clase.


In [ ]:
print("INTERPRETACIÓN GENERAL")
print("----------------------")

if sil_score >= 0.50:
    print("Los clusters presentan una separación buena o alta.")
elif sil_score >= 0.25:
    print("Los clusters presentan una separación moderada o aceptable.")
else:
    print("Los clusters presentan una separación débil y podrían mejorarse.")

if db_score < 1:
    print("El índice Davies-Bouldin sugiere grupos relativamente compactos y diferenciados.")
else:
    print("El índice Davies-Bouldin sugiere revisar la calidad de separación entre clusters.")

print("Recordatorio:")
print("- En Silhouette, mientras más alto, mejor.")
print("- En Davies-Bouldin, mientras más bajo, mejor.")
print("- En Calinski-Harabasz, mientras más alto, mejor.")

## Paso 17: Caracterizar los clusters

Ahora calcularemos los promedios de cada variable por cluster.

Aquí ocurre la parte más importante del modelo descriptivo:

> **interpretar qué representa cada grupo**


In [ ]:
perfil_clusters = df.groupby("CLUSTER").mean(numeric_only=True)
perfil_clusters

## Paso 18: Visualizar los clusters en dos dimensiones

Como la representación latente tiene 3 dimensiones, tomaremos dos para visualizar los grupos de manera simple.


In [ ]:
plt.figure(figsize=(8,6))
plt.scatter(X_latent[:,0], X_latent[:,1], c=clusters)
plt.title("Visualización de clusters en espacio latente")
plt.xlabel("Dimensión latente 1")
plt.ylabel("Dimensión latente 2")
plt.show()

## Paso 19: Ejemplo de interpretación de perfiles

Con la tabla de promedios puedes generar descripciones como:

- **Cluster 0**: trabajadores con mayor nivel de estrés y mayor consumo de café
- **Cluster 1**: trabajadores con menos café y más horas de sueño
- **Cluster 2**: trabajadores con patrón intermedio o asociado a reuniones y trabajo remoto

Estas etiquetas deben construirse observando los datos, no inventarse sin evidencia.


## Paso 20: Conclusión del ejercicio

### ¿Por qué esta técnica es adecuada?
La técnica aplicada es adecuada porque:

- el problema es **descriptivo**
- no existe variable objetivo
- queremos descubrir grupos ocultos
- la red neuronal autoencoder aprende una representación compacta del comportamiento
- el clustering permite segmentar perfiles similares

### ¿Cómo sabemos si el modelo está operando correctamente?
Lo evaluamos mediante:

- **loss** y **validation loss** del autoencoder
- **silhouette score**
- **Davies-Bouldin index**
- **Calinski-Harabasz score**
- interpretación coherente de los clusters

### Idea final
En modelos descriptivos no preguntamos:

> “¿predijo correctamente?”

Sino:

> “¿agrupó de manera coherente y útil?”

Ese es el criterio correcto en este tipo de análisis.


## Preguntas para los estudiantes

1. ¿Por qué este caso corresponde a un modelo descriptivo?
2. ¿Qué función cumple la red neuronal autoencoder?
3. ¿Por qué no usamos accuracy o matriz de confusión en este caso?
4. ¿Qué indica el loss del autoencoder?
5. ¿Qué significa un silhouette score alto?
6. ¿Qué significa un Davies-Bouldin bajo?
7. ¿Qué características distinguen a cada cluster?
8. ¿Qué decisiones podría tomar la empresa a partir de estos grupos?
